In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import torch
import torch.nn as nn
import torch.optim as optim

# 1. 加载数据
train = pd.read_csv('train.csv')  # 训练数据
test = pd.read_csv('test.csv')    # 测试数据

In [2]:
train['days'] = pd.to_datetime(train['date'])
test['days'] = pd.to_datetime(test['date'])
base_date = pd.to_datetime('2010-01-01')
train['days'] = (train['days'] - base_date).dt.days
test['days'] = (test['days'] - base_date).dt.days
train['days'].head()

0    0
1    0
2    0
3    0
4    0
Name: days, dtype: int64

In [3]:
train = pd.get_dummies(train, columns=['country'], drop_first=True)  # 独热编码
test = pd.get_dummies(test, columns=['country'], drop_first=True)
train = pd.get_dummies(train, columns=['store'], drop_first=True)  # 独热编码
test = pd.get_dummies(test, columns=['store'], drop_first=True)

In [4]:
train['num_sold'] = train['num_sold'].fillna(0)

In [5]:
features = [ 'country_Finland', 'country_Italy', 'country_Norway', 'country_Kenya', 'country_Singapore', 'store_Premium Sticker Mart', 'store_Stickers for Less']
X = train[features]
train['zero'] = train['num_sold'] == 0
Y = train['zero']
Y.head()

0     True
1    False
2    False
3    False
4    False
Name: zero, dtype: bool

In [6]:
X_train, X_val, y_train, y_val = train_test_split(X, Y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train.to_list(), dtype=torch.float32)
X_test = torch.tensor(X_val, dtype=torch.float32)
y_test = torch.tensor(y_val.to_list(), dtype=torch.float32)
y_test

tensor([0., 0., 1.,  ..., 0., 0., 0.])

In [7]:
class LinearSVM(nn.Module):
    def __init__(self, input_dim):
        super(LinearSVM, self).__init__()
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        return self.linear(x).squeeze()

def hinge_loss(y_pred, y_true, epsilon=0.1):
    return torch.mean(torch.clamp(1 - y_pred * y_true, min=0))


In [8]:
imput_dim = X_train.shape[1]
model = LinearSVM(imput_dim)

optimizer = optim.SGD(model.parameters(), lr=0.1)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

num_epochs = 5000
for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()

    outputs = model(X_train).squeeze()

    loss = hinge_loss(outputs, y_train)
    loss.backward()
    optimizer.step()

    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

Epoch [10/5000], Loss: 0.9811
Epoch [20/5000], Loss: 0.9774
Epoch [30/5000], Loss: 0.9744
Epoch [40/5000], Loss: 0.9731
Epoch [50/5000], Loss: 0.9718
Epoch [60/5000], Loss: 0.9705
Epoch [70/5000], Loss: 0.9692
Epoch [80/5000], Loss: 0.9679
Epoch [90/5000], Loss: 0.9666
Epoch [100/5000], Loss: 0.9660
Epoch [110/5000], Loss: 0.9653
Epoch [120/5000], Loss: 0.9646
Epoch [130/5000], Loss: 0.9640
Epoch [140/5000], Loss: 0.9639
Epoch [150/5000], Loss: 0.9637
Epoch [160/5000], Loss: 0.9636
Epoch [170/5000], Loss: 0.9635
Epoch [180/5000], Loss: 0.9633
Epoch [190/5000], Loss: 0.9632
Epoch [200/5000], Loss: 0.9630
Epoch [210/5000], Loss: 0.9629
Epoch [220/5000], Loss: 0.9627
Epoch [230/5000], Loss: 0.9626
Epoch [240/5000], Loss: 0.9624
Epoch [250/5000], Loss: 0.9623
Epoch [260/5000], Loss: 0.9622
Epoch [270/5000], Loss: 0.9620
Epoch [280/5000], Loss: 0.9619
Epoch [290/5000], Loss: 0.9617
Epoch [300/5000], Loss: 0.9616
Epoch [310/5000], Loss: 0.9614
Epoch [320/5000], Loss: 0.9614
Epoch [330/5000],

In [17]:
model.eval()
# def mse_loss(y_pred, y_true):
#     return torch.mean((y_pred - y_true) ** 2)
#
# with torch.no_grad():
#     outputs = model(X_test).squeeze()
#     mse = mse_loss(outputs, y_test)
#     print(f'MSE: {mse.item():.4f}')
with torch.no_grad():
    outputs = model(X_test).squeeze().floor()
    predictions = torch.sign(outputs)
    predictions = predictions.round()
    accuracy = torch.mean((predictions == y_test).float())
    print(f'Test Accuracy: {accuracy.item() * 100:.2f}%')

Test Accuracy: 53.90%


In [18]:
# 假设 y_test 是真实值，outputs 是预测值
y_test_np = y_test.numpy()  # 将真实值转换为 NumPy 数组
outputs_np = outputs.numpy()  # 将预测值转换为 NumPy 数组


# 反标准化预测值

# 创建 DataFrame 来存储真实值和反标准化后的预测值
results = pd.DataFrame({
    'True Value': y_test_np,  # 真实值
    'Predicted Value (Original Scale)': outputs_np  # 反标准化后的预测值
})

# 打印表格
print(results)

       True Value  Predicted Value (Original Scale)
0             0.0                               0.0
1             0.0                               0.0
2             1.0                               1.0
3             0.0                              -1.0
4             0.0                               1.0
...           ...                               ...
46021         0.0                               1.0
46022         0.0                               0.0
46023         0.0                               1.0
46024         0.0                               0.0
46025         0.0                               0.0

[46026 rows x 2 columns]
